In [ ]:
import torch
from d2l import torch as d2l
from torch import nn

In [ ]:
#核心创新在于layer的上一层抽象block的引入。

def vgg_block(num_convs,in_channels,out_channels):
    layers=[]
    for i in range(num_convs):
        layers.append(nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1))
        layers.append(nn.ReLU())
        in_channels=out_channels
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
    return nn.Sequential(*layers)

def vgg(conv_arch):
    blocks=[]
    in_channels=1
    for pairs in conv_arch:
        blocks.append(vgg_block(pairs[0],in_channels,pairs[1]))
        in_channels=pairs[1]
    return nn.Sequential(*blocks,
                         nn.Flatten(),
                         nn.Linear(conv_arch[-1][1]*7*7,4096),nn.ReLU(),nn.Dropout(0.5),
                         nn.Linear(4096,4096),nn.ReLU(),nn.Dropout(0.5),
                         nn.Linear(4096,10)
    )

In [ ]:
#torch 高级API版
batch_size=128
lr=0.05
num_epochs=10
conv_arch=[(1,16),(1,32),(2,64),(2,128),(2,128)]
net=vgg(conv_arch)
X = torch.randn(size=(1, 1, 224, 224)) 
for blk in net:  
    X = blk(X) 
    print(blk.__class__.__name__,'output shape:\t',X.shape)

def init_weight(m):
    if type(m)==nn.Linear or type(m)==nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)

net.apply(init_weight)

loss=nn.CrossEntropyLoss()

updater=torch.optim.SGD(net.parameters(),lr=lr)

train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=224)


In [ ]:
def evaluate_accuracy_gpu(net, data_iter, device=None): #@save
    """使用GPU计算模型在数据集上的精度"""
    if isinstance(net, nn.Module):
        net.eval()  # 设置为评估模式
        if not device:
            device = next(iter(net.parameters())).device
    # 正确预测的数量，总预测的数量
    metric = d2l.Accumulator(2)
    with torch.no_grad():
        for X, y in data_iter:
            if isinstance(X, list):
                # BERT微调所需的（之后将介绍）
                X = [x.to(device) for x in X]
            else:
                X = X.to(device)
            y = y.to(device)
            metric.add(d2l.accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

In [ ]:
def train_ch6(net, train_iter, test_iter, optimizer,loss,num_epochs, device):
    print('training on', device)
    net.to(device)
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs],
                            legend=['train loss', 'train acc', 'test acc'])
    timer, num_batches = d2l.Timer(), len(train_iter)
    for epoch in range(num_epochs):
        # 训练损失之和，训练准确率之和，样本数
        metric = d2l.Accumulator(3)
        net.train()
        for i, (X, y) in enumerate(train_iter):
            timer.start()
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.accuracy(y_hat, y), X.shape[0])
            timer.stop()
            train_l = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (train_l, train_acc, None))
        test_acc = evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch + 1, (None, None, test_acc))
    print(f'loss {train_l:.3f}, train acc {train_acc:.3f}, '
          f'test acc {test_acc:.3f}')
    print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec '
          f'on {str(device)}')

In [ ]:
train_ch6(net,train_iter,test_iter,updater,loss,num_epochs,torch.device('cuda'))